# <center> **<span style="font-size:80px;">Físicos</span>** </center>
# 🌍 MODELO PREDICTIVO Y AUDITORÍA DE FRAUDES POR BARRIOS
---
Este notebook une datos reales de Alicante (agua, clima, satélite, VT) para predecir el consumo mediante Fourier + Machine Learning. Además, incluye un sistema de auditoría que detecta anomalías y asigna porcentajes de culpabilidad para diferenciar entre causas climáticas/turísticas y posibles fraudes o fugas.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import scipy.stats as stats
import sys
import os

from scipy.optimize import curve_fit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Tu configuración de rutas del proyecto
sys.path.append(os.path.abspath(os.path.join("..")))
try:
    from src.config import DatasetKeys, Paths
    Paths.init_project()
    print("✅ Entorno del proyecto cargado correctamente.")
except ImportError:
    print("⚠️ No se pudo cargar src.config. Asegúrate de estar en el directorio correcto.")

In [ ]:
# 1. CARGA DE DATOS REALES Y AGRUPACIÓN POR BARRIO
print("1. Procesando datos por barrio...")

df_agua  = pd.read_csv(Paths.PROC_CSV_AMAEM_SCALED)
df_clima = pd.read_csv(Paths.AEMET_CLIMA_BARRIOS)
df_ndvi  = pd.read_csv(Paths.SENTINEL_NDVI)

df_agua['fecha_mes'] = pd.to_datetime(df_agua['fecha']).dt.to_period('M').astype(str)
df_clima['fecha_mes'] = df_clima['fecha'].astype(str)

df_ts = df_agua.groupby(['fecha_mes', 'barrio']).agg({
    'consumo': 'sum',
    'num_vt_barrio': 'mean',
    'porcentaje_vt_barrio %': 'mean',
    'ocupaciones_vt_prov': 'mean',
    'pernoctaciones_vt_prov': 'mean'
}).reset_index()

df_clima_ciudad = df_clima.groupby('fecha_mes')[['tm_mes', 'p_mes']].mean().reset_index()

df_final = pd.merge(df_ts, df_clima_ciudad, on='fecha_mes', how='left')
df_final = pd.merge(df_final, df_ndvi, on='fecha_mes', how='left')
df_final = df_final.dropna()
print(f"Total de registros listos: {len(df_final)} (Meses x Barrios)")

In [ ]:
df_final.columns

In [ ]:
# 2. MOTOR MATEMÁTICO (FOURIER) - ¡ENTRENADO BARRIO POR BARRIO!
print("2. Calculando la estacionalidad física independiente para cada barrio...")

def modelo_fourier(t, m, c, a1, b1, a2, b2):
    w = 2 * np.pi / 12
    return (m * t + c) + (a1 * np.cos(w * t) + b1 * np.sin(w * t)) + (a2 * np.cos(2 * w * t) + b2 * np.sin(2 * w * t))

df_final['prediccion_fourier'] = 0.0

for barrio in df_final['barrio'].unique():
    mask = df_final['barrio'] == barrio
    y_barrio = df_final.loc[mask, 'consumo'].values
    t_barrio = np.arange(len(y_barrio))
    
    try:
        coef, _ = curve_fit(modelo_fourier, t_barrio, y_barrio, p0=[0, np.mean(y_barrio), 1000, 1000, 100, 100], maxfev=10000)
        df_final.loc[mask, 'prediccion_fourier'] = modelo_fourier(t_barrio, *coef)
    except:
        df_final.loc[mask, 'prediccion_fourier'] = np.mean(y_barrio)

print("¡Curvas físicas calculadas con éxito!")

In [ ]:
# 3. MACHINE LEARNING (PREDICCIÓN DE COMPORTAMIENTO)
print("3. Entrenando IA global con contexto geográfico...")
df_final['residuo'] = df_final['consumo'] - df_final['prediccion_fourier']

df_ml = pd.get_dummies(df_final, columns=['barrio'])

exogenas = ['tm_mes', 'p_mes', 'ndvi_satelite', 
            'num_vt_barrio', 'porcentaje_vt_barrio %', 
            'ocupaciones_vt_prov', 'pernoctaciones_vt_prov']

columnas_barrios = [col for col in df_ml.columns if col.startswith('barrio_')]
X = df_ml[exogenas + columnas_barrios]
y = df_ml['residuo']

ml_model = RandomForestRegressor(n_estimators=100, random_state=42)
ml_model.fit(X, y)
df_final['impacto_exogeno'] = ml_model.predict(X)

df_final['prediccion_hibrida'] = df_final['prediccion_fourier'] + df_final['impacto_exogeno']

print("\n--- MÉTRICAS GLOBALES DE LA CIUDAD ---")
print(f"Precisión de Fourier (Solo matemáticas): {r2_score(df_final['consumo'], df_final['prediccion_fourier']):.4f}")
print(f"Precisión Híbrida (IA + Exógenas + Barrios): {r2_score(df_final['consumo'], df_final['prediccion_hibrida']):.4f}")

In [ ]:
# 4. DETECCIÓN DE ANOMALÍAS Y ATRIBUCIÓN DE CAUSAS (% DE CULPABILIDAD)
print("\n4. Analizando anomalías y calculando culpables...")

sospechosos = {
    'tm_mes': 'Calor',
    'p_mes': 'Falta_Lluvia',
    'ndvi_satelite': 'Sequedad_Jardines',
    'pernoctaciones_vt_prov': 'Turismo_Masivo'
}

# Z-Score para las variables climáticas y turísticas
for var in sospechosos.keys():
    df_final[f'z_{var}'] = df_final.groupby('barrio')[var].transform(lambda x: stats.zscore(x, ddof=1)).fillna(0)
    df_final[f'peso_{var}'] = df_final[f'z_{var}'].abs()

# Z-Score del Error (lo que la IA no supo explicar)
df_final['error_final'] = df_final['consumo'] - df_final['prediccion_hibrida']
df_final['z_error_final'] = df_final.groupby('barrio')['error_final'].transform(lambda x: stats.zscore(x, ddof=1)).fillna(0)
df_final['peso_Desconocido'] = df_final['z_error_final'].abs()

columnas_pesos = [f'peso_{var}' for var in sospechosos.keys()] + ['peso_Desconocido']
df_final['suma_pesos'] = df_final[columnas_pesos].sum(axis=1)
df_final['suma_pesos'] = df_final['suma_pesos'].replace(0, 1)

# Convertir a porcentajes
for var, nombre_bonito in sospechosos.items():
    df_final[f'%_{nombre_bonito}'] = (df_final[f'peso_{var}'] / df_final['suma_pesos']) * 100

df_final['%_Fraude_o_Fuga'] = (df_final['peso_Desconocido'] / df_final['suma_pesos']) * 100

# Filtrar solo las Alertas Rojas
limite_anomalia = 1.0
anomalias_graves = df_final[df_final['z_error_final'] > limite_anomalia].copy()

columnas_vista = ['fecha_mes', 'barrio', 'consumo', 'prediccion_hibrida', 
                  '%_Calor', '%_Falta_Lluvia', '%_Sequedad_Jardines', '%_Turismo_Masivo', '%_Fraude_o_Fuga']

reporte_anomalias = anomalias_graves[columnas_vista].sort_values(by='%_Fraude_o_Fuga', ascending=False)

for col in columnas_vista[4:]:
    reporte_anomalias[col] = reporte_anomalias[col].round(1).astype(str) + '%'

print(f"\n🚨 ¡Se han encontrado {len(reporte_anomalias)} picos de consumo masivo!")
print("Aquí está el reparto de culpabilidad de los casos más sospechosos:\n")
print(reporte_anomalias.head(10).to_string(index=False))

# Exportar
reporte_anomalias.to_csv('informe_fraudes_y_causas.csv', index=False)
print("\n✅ Archivo 'informe_fraudes_y_causas.csv' exportado.")

In [ ]:
# 5. GRÁFICA PARA LA PRESENTACIÓN (Mostramos 1 barrio como ejemplo)
barrio_ejemplo = df_final['barrio'].unique()[0] 
df_plot = df_final[df_final['barrio'] == barrio_ejemplo]

plt.figure(figsize=(14, 6))
plt.plot(df_plot['fecha_mes'], df_plot['consumo'], 'ko-', label=f'Consumo Real ({barrio_ejemplo})', linewidth=2)
plt.plot(df_plot['fecha_mes'], df_plot['prediccion_fourier'], 'b--', alpha=0.5, label='Física (Curva base del barrio)')
plt.plot(df_plot['fecha_mes'], df_plot['prediccion_hibrida'], 'r-', linewidth=3, label='Híbrido (Matemática + Machine Learning)')

plt.title(f'Predicción Hídrica Hiper-segmentada: Impacto Exógeno en {barrio_ejemplo}', fontsize=16)
plt.xticks(rotation=45)
plt.ylabel('Consumo de Agua')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()